##Load Dataset

In [2]:
import pandas as pd
import numpy as np
import html

df = pd.read_csv("movies.csv")

print("Dataset shape:", df.shape)
df.head()

Dataset shape: (1794, 15)


,year,imdb,title,test,clean_test,binary,budget,domgross,intgross,code,budget_2013$,domgross_2013$,intgross_2013$,period code,decade code
0,2013,tt1711425,21 &amp; Over,notalk,notalk,FAIL,13000000,25682380.0,42195766.0,2013FAIL,13000000,25682380.0,42195766.0,1.0,1.0
1,2012,tt1343727,Dredd 3D,ok-disagree,ok,PASS,45000000,13414714.0,40868994.0,2012PASS,45658735,13611086.0,41467257.0,1.0,1.0
2,2013,tt2024544,12 Years a Slave,notalk-disagree,notalk,FAIL,20000000,53107035.0,158607035.0,2013FAIL,20000000,53107035.0,158607035.0,1.0,1.0
3,2013,tt1272878,2 Guns,notalk,notalk,FAIL,61000000,75612460.0,132493015.0,2013FAIL,61000000,75612460.0,132493015.0,1.0,1.0
4,2013,tt0453562,42,men,men,FAIL,40000000,95020213.0,95020213.0,2013FAIL,40000000,95020213.0,95020213.0,1.0,1.0


##Initial data review

In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1794 entries, 0 to 1793
Data columns (total 15 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   year            1794 non-null   int64  
 1   imdb            1794 non-null   object 
 2   title           1794 non-null   object 
 3   test            1794 non-null   object 
 4   clean_test      1794 non-null   object 
 5   binary          1794 non-null   object 
 6   budget          1794 non-null   int64  
 7   domgross        1777 non-null   float64
 8   intgross        1783 non-null   float64
 9   code            1794 non-null   object 
 10  budget_2013$    1794 non-null   int64  
 11  domgross_2013$  1776 non-null   float64
 12  intgross_2013$  1783 non-null   float64
 13  period code     1615 non-null   float64
 14  decade code     1615 non-null   float64
dtypes: float64(6), int64(3), object(6)
memory usage: 210.4+ KB


##Missing values and data quality check

In [4]:
print("Rows:", len(df))
print("Columns:", len(df.columns))
print("Duplicate rows:", df.duplicated().sum())
print("Duplicate IMDb IDs:", df["imdb"].duplicated().sum())
print("Year range:", df["year"].min(), "-", df["year"].max())

print("\nMissing values:")
print(df.isna().sum())

Rows: 1794
Columns: 15
Duplicate rows: 0
Duplicate IMDb IDs: 0
Year range: 1970 - 2013

Missing values:
year                0
imdb                0
title               0
test                0
clean_test          0
binary              0
budget              0
domgross           17
intgross           11
code                0
budget_2013$        0
domgross_2013$     18
intgross_2013$     11
period code       179
decade code       179
dtype: int64


In [5]:
print("Duplicate rows:", df.duplicated().sum())
print("Duplicate IMDb IDs:", df["imdb"].duplicated().sum())

print(
    "Year range:",
    df["year"].min(),
    "-",
    df["year"].max()
)

print("\nMissing values:")
display(
    df.isna()
      .sum()
      .sort_values(ascending=False)
      .to_frame("Missing values")
)

Duplicate rows: 0
Duplicate IMDb IDs: 0
Year range: 1970 - 2013

Missing values:


,Missing values
period code,179
decade code,179
domgross_2013$,18
domgross,17
intgross,11
intgross_2013$,11
year,0
budget,0
binary,0
clean_test,0


##Preprocessing

In [6]:
clean_df = df.copy()

# Clean movie titles
clean_df["TITLE_CLEAN"] = (
    clean_df["title"]
    .apply(html.unescape)
)

# Create decade for the full 1970-2013 period
clean_df["DECADE"] = (
    clean_df["year"] // 10 * 10
).astype(str) + "s"

# Standardize Bechdel result
clean_df["BECHDEL_RESULT"] = (
    clean_df["binary"]
    .str.strip()
    .str.upper()
)

# Financial variables
financial_cols = [
    "budget_2013$",
    "domgross_2013$",
    "intgross_2013$"
]

for col in financial_cols:
    clean_df[col] = pd.to_numeric(
        clean_df[col],
        errors="coerce"
    )

# Derived financial metrics
clean_df["PROFIT_2013"] = (
    clean_df["intgross_2013$"]
    - clean_df["budget_2013$"]
)

clean_df["ROI_2013"] = (
    clean_df["PROFIT_2013"]
    / clean_df["budget_2013$"]
)

clean_df["ROI_2013"] = (
    clean_df["ROI_2013"]
    .replace([np.inf, -np.inf], np.nan)
)

##Validation after preprocessing

In [7]:
print("Original records:", len(df))
print("Records after preprocessing:", len(clean_df))

print("\nBechdel results:")
print(clean_df["BECHDEL_RESULT"].value_counts())

print("\nDecades:")
print(clean_df["DECADE"].value_counts().sort_index())

print("\nMissing values in derived financial values:")
print(
    clean_df[
        [
            "budget_2013$",
            "domgross_2013$",
            "intgross_2013$",
            "PROFIT_2013",
            "ROI_2013"
        ]
    ].isna().sum()
)

Original records: 1794
Records after preprocessing: 1794

Bechdel results:
BECHDEL_RESULT
FAIL    991
PASS    803
Name: count, dtype: int64

Decades:
DECADE
1970s     54
1980s    125
1990s    337
2000s    840
2010s    438
Name: count, dtype: int64

Missing values in derived financial values:
budget_2013$       0
domgross_2013$    18
intgross_2013$    11
PROFIT_2013       11
ROI_2013          11
dtype: int64


##Export

In [8]:
clean_df.to_csv(
    "movies_Cleaned.csv",
    index=False
)

print("Saved: movies_Cleaned.csv")
print("Records:", len(clean_df))
print("Variables:", len(clean_df.columns))

Saved: movies_Cleaned.csv
Records: 1794
Variables: 20


In [9]:
from google.colab import files

files.download(
    "/content/movies_Cleaned.csv"
)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

##Data for web prototype

In [10]:
web_df = clean_df[
    [
        "year",
        "imdb",
        "TITLE_CLEAN",
        "clean_test",
        "BECHDEL_RESULT",
        "budget_2013$",
        "domgross_2013$",
        "intgross_2013$",
        "PROFIT_2013",
        "ROI_2013",
        "DECADE"
    ]
].copy()

In [11]:
print("Web records:", len(web_df))
print("Web variables:", len(web_df.columns))

print("\nYear range:")
print(web_df["year"].min(), "-", web_df["year"].max())

print("\nBechdel results:")
print(web_df["BECHDEL_RESULT"].value_counts())

print("\nMissing values:")
print(
    web_df.isna()
    .sum()
    .sort_values(ascending=False)
)

Web records: 1794
Web variables: 11

Year range:
1970 - 2013

Bechdel results:
BECHDEL_RESULT
FAIL    991
PASS    803
Name: count, dtype: int64

Missing values:
domgross_2013$    18
ROI_2013          11
PROFIT_2013       11
intgross_2013$    11
TITLE_CLEAN        0
year               0
imdb               0
budget_2013$       0
BECHDEL_RESULT     0
clean_test         0
DECADE             0
dtype: int64


In [12]:
web_df.to_json(
    "screenequity.json",
    orient="records",
    indent=2
)

In [13]:
files.download(
    "/content/screenequity.json"
)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>